In [1]:
import sys
sys.path.append("..")
import torch
from model import GPTModel, generate
from transformers import GPT2LMHeadModel
import tiktoken

# Download HF weights
hf_model = GPT2LMHeadModel.from_pretrained("gpt2")
hf_sd = hf_model.state_dict()

# Your model — using config dict
cfg = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.0,
    "qkv_bias": True  # GPT-2 uses biases
}

model = GPTModel(cfg)

# Compare weight names
print("=== HF names ===")
for k in list(hf_sd.keys())[:12]:
    print(k, hf_sd[k].shape)

print("\n=== Your names ===")
for k in list(model.state_dict().keys())[:12]:
    print(k, model.state_dict()[k].shape)

/home/rishie/miniconda3/envs/llm-from-scratch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████████████████████████████| 148/148 [00:00<00:00, 1086.89it/s]


=== HF names ===
transformer.wte.weight torch.Size([50257, 768])
transformer.wpe.weight torch.Size([1024, 768])
transformer.h.0.ln_1.weight torch.Size([768])
transformer.h.0.ln_1.bias torch.Size([768])
transformer.h.0.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.0.attn.c_attn.bias torch.Size([2304])
transformer.h.0.attn.c_proj.weight torch.Size([768, 768])
transformer.h.0.attn.c_proj.bias torch.Size([768])
transformer.h.0.ln_2.weight torch.Size([768])
transformer.h.0.ln_2.bias torch.Size([768])
transformer.h.0.mlp.c_fc.weight torch.Size([768, 3072])
transformer.h.0.mlp.c_fc.bias torch.Size([3072])

=== Your names ===
tok_emb.weight torch.Size([50257, 768])
pos_emb.weight torch.Size([1024, 768])
trf_blocks.0.att.mask torch.Size([1024, 1024])
trf_blocks.0.att.W_query.weight torch.Size([768, 768])
trf_blocks.0.att.W_query.bias torch.Size([768])
trf_blocks.0.att.W_key.weight torch.Size([768, 768])
trf_blocks.0.att.W_key.bias torch.Size([768])
trf_blocks.0.att.W_value.weight tor

In [2]:
from model import GPTModel
help(GPTModel.__init__)

Help on function __init__ in module model:

__init__(self, cfg)
    Initialize internal Module state, shared by both nn.Module and ScriptModule.



In [3]:
def load_gpt2_weights(model, hf_sd):
    """Map HuggingFace GPT-2 weights into our architecture."""
    sd = model.state_dict()
    
    # Embeddings — direct copy
    sd["tok_emb.weight"] = hf_sd["transformer.wte.weight"]
    sd["pos_emb.weight"] = hf_sd["transformer.wpe.weight"]
    
    for i in range(12):  # 12 layers
        hf = f"transformer.h.{i}"
        ours = f"trf_blocks.{i}"
        
        # LayerNorm 1
        sd[f"{ours}.norm1.scale"] = hf_sd[f"{hf}.ln_1.weight"]
        sd[f"{ours}.norm1.shift"] = hf_sd[f"{hf}.ln_1.bias"]
        
        # LayerNorm 2
        sd[f"{ours}.norm2.scale"] = hf_sd[f"{hf}.ln_2.weight"]
        sd[f"{ours}.norm2.shift"] = hf_sd[f"{hf}.ln_2.bias"]
        
        # Attention: HF combines QKV into one [768, 2304] matrix
        # Split into 3 chunks and transpose (Conv1D -> Linear)
        qkv_w = hf_sd[f"{hf}.attn.c_attn.weight"]  # [768, 2304]
        q_w, k_w, v_w = qkv_w.split(768, dim=1)
        sd[f"{ours}.att.W_query.weight"] = q_w.T
        sd[f"{ours}.att.W_key.weight"] = k_w.T
        sd[f"{ours}.att.W_value.weight"] = v_w.T
        
        qkv_b = hf_sd[f"{hf}.attn.c_attn.bias"]  # [2304]
        q_b, k_b, v_b = qkv_b.split(768)
        sd[f"{ours}.att.W_query.bias"] = q_b
        sd[f"{ours}.att.W_key.bias"] = k_b
        sd[f"{ours}.att.W_value.bias"] = v_b
        
        # Output projection — transpose Conv1D -> Linear
        sd[f"{ours}.att.W_o.weight"] = hf_sd[f"{hf}.attn.c_proj.weight"].T
        sd[f"{ours}.att.W_o.bias"] = hf_sd[f"{hf}.attn.c_proj.bias"]
        
        # Feed-forward — transpose Conv1D -> Linear
        sd[f"{ours}.ff.layers.0.weight"] = hf_sd[f"{hf}.mlp.c_fc.weight"].T
        sd[f"{ours}.ff.layers.0.bias"] = hf_sd[f"{hf}.mlp.c_fc.bias"]
        sd[f"{ours}.ff.layers.2.weight"] = hf_sd[f"{hf}.mlp.c_proj.weight"].T
        sd[f"{ours}.ff.layers.2.bias"] = hf_sd[f"{hf}.mlp.c_proj.bias"]
    
    # Final LayerNorm
    sd["final_norm.scale"] = hf_sd["transformer.ln_f.weight"]
    sd["final_norm.shift"] = hf_sd["transformer.ln_f.bias"]
    
    # Output head
    sd["out_head.weight"] = hf_sd["transformer.wte.weight"]  # weight tying
    
    model.load_state_dict(sd)
    return model

model = load_gpt2_weights(model, hf_sd)
print("Weights loaded!")

Weights loaded!
